In [21]:
# Install dependencies
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
import pickle

In [22]:
# Load in dataframe, then sort from oldest to newest
df = pd.read_csv('./games.csv')
df = df.sort_values("GAME_DATE_EST")

# View first five rows of the dataset
df.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
15681,2003-10-05,10300001,Final,1610612762,1610612742,2003,1610612762,90.0,0.457,0.735,...,23.0,41.0,1610612742,85.0,0.447,0.500,0.250,20.0,38.0,1
15680,2003-10-06,10300002,Final,1610612763,1610612749,2003,1610612763,105.0,0.494,0.618,...,25.0,48.0,1610612749,94.0,0.427,0.700,0.154,20.0,43.0,1
15672,2003-10-07,10300006,Final,1610612747,1610612744,2003,1610612747,NaN,NaN,NaN,...,NaN,NaN,1610612744,NaN,NaN,NaN,NaN,NaN,NaN,0
15679,2003-10-07,10300009,Final,1610612758,1610612746,2003,1610612758,101.0,0.467,0.871,...,19.0,39.0,1610612746,82.0,0.368,0.609,0.364,13.0,50.0,1
15678,2003-10-07,10300005,Final,1610612757,1610612745,2003,1610612757,104.0,0.527,0.657,...,22.0,33.0,1610612745,80.0,0.470,0.667,0.333,10.0,37.0,1


In [23]:
print(df.columns.tolist())
print(df.head())

['GAME_DATE_EST', 'GAME_ID', 'GAME_STATUS_TEXT', 'HOME_TEAM_ID', 'VISITOR_TEAM_ID', 'SEASON', 'TEAM_ID_home', 'PTS_home', 'FG_PCT_home', 'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home', 'TEAM_ID_away', 'PTS_away', 'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away', 'REB_away', 'HOME_TEAM_WINS']
      GAME_DATE_EST   GAME_ID GAME_STATUS_TEXT  HOME_TEAM_ID  VISITOR_TEAM_ID  \
15681    2003-10-05  10300001            Final    1610612762       1610612742   
15680    2003-10-06  10300002            Final    1610612763       1610612749   
15672    2003-10-07  10300006            Final    1610612747       1610612744   
15679    2003-10-07  10300009            Final    1610612758       1610612746   
15678    2003-10-07  10300005            Final    1610612757       1610612745   

       SEASON  TEAM_ID_home  PTS_home  FG_PCT_home  FT_PCT_home  ...  \
15681    2003    1610612762      90.0        0.457        0.735  ...   
15680    2003    1610612763     105.0        0.494        0.6

In [24]:
# List out column names
print(df.columns)

# Choose feature columns
columns = ["PTS", "FG_PCT", "FG3_PCT", "FT_PCT", "AST", "REB"]

# Set window for rolling averages (needed for predictions)
WINDOW = 20

# Reformat: one row per team per game (for home and away)
home = df[["GAME_DATE_EST", "GAME_ID", "HOME_TEAM_ID"] + [f"{c}_home" for c in columns]].copy()
home.columns = ["DATE", "GAME_ID", "TEAM_ID"] + columns
home["HOME"] = 1

away = df[["GAME_DATE_EST", "GAME_ID", "VISITOR_TEAM_ID"] + [f"{c}_away" for c in columns]].copy()
away.columns = ["DATE", "GAME_ID", "TEAM_ID"] + columns
away["HOME"] = 0

# Combine these dataframes into one using .concat()
# Ends up giving us a table with two rows per game, one for home, one for away
games = pd.concat([home, away]).sort_values("DATE").reset_index(drop = True)

# Computes rolling average of stat columns up to a certain point (for each team)
games[columns] = (
    games.groupby("TEAM_ID")[columns]
    .transform(lambda x: x.shift(1).rolling(WINDOW, min_periods = 3).mean())
)

# Drops rows where there are less than three games of precedent
games = games.dropna(subset = columns)

Index(['GAME_DATE_EST', 'GAME_ID', 'GAME_STATUS_TEXT', 'HOME_TEAM_ID',
       'VISITOR_TEAM_ID', 'SEASON', 'TEAM_ID_home', 'PTS_home', 'FG_PCT_home',
       'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home', 'TEAM_ID_away',
       'PTS_away', 'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away',
       'REB_away', 'HOME_TEAM_WINS'],
      dtype='object')


In [25]:
# Creates separate dfs for home and away (with rolling averages)
home_stats = games[games["HOME"] == 1].set_index("GAME_ID")[columns].add_prefix("home_")
away_stats = games[games["HOME"] == 0].set_index("GAME_ID")[columns].add_prefix("away_")

# Combines those separated dfs using pre_set index (GAME_ID)
# Thus, one row now represents one game, with a "home_/away_" col for each stat
features = home_stats.join(away_stats, how = "inner")

# Takes diffs between home and away for each stat, creating new columns
for col in columns:
    features[f"DIFF_{col}"] = features[f"home_{col}"] - features[f"away_{col}"]

# Adds in label for model prediction, dropping any unobserved values
# In this case, label is whether or not home time wins
features = features.join(df.set_index("GAME_ID")["HOME_TEAM_WINS"])
features = features.dropna()

In [26]:
# Add date back to features
features["DATE"] = features.index.map(df.set_index("GAME_ID")["GAME_DATE_EST"])

# Define our features and our label and split
X = features[[f"DIFF_{col}" for col in columns]]
y = features["HOME_TEAM_WINS"]

# Obtain train-test-split by time instead of randomly
# Not random because it makes no sense to predict a result for a game in March
# by using a game from May, since that game in May already contains info about
# that March game in its rolling averages as calculated earlier.
# This date splits our data 80/20
split = "2016-12-01"

# Get X and y splits
train_mask = features["DATE"] < split
test_mask = features["DATE"] >= split
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

# Scale the feature data to standardize unit of measure across stats
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize the model with max iteration limit
model = LogisticRegression(max_iter = 1000)

# Fit the model using scaled X data
model.fit(X_train_scaled, y_train)

# Evaluates the accuracy of the model using test data
print(f"Accuracy: {accuracy_score(y_test, model.predict(X_test_scaled)):.3f}")
print(f"Log Loss: {log_loss(y_test, model.predict_proba(X_test_scaled)):.3f}")

Accuracy: 0.608
Log Loss: 0.655


In [27]:
# Saves the model and the scaler for the web app
pickle.dump(model, open("nba_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

In [28]:
# Import the teams dataset to associate teams with team IDs
df_teams = pd.read_csv('./teams.csv')

# Get the column names and visualize first few rows of data
#print(df_teams.columns.tolist())
#print(df_teams.head())

# Create new column that combines city with nickname to get team
df_teams["FULL_NAME"] = df_teams["CITY"] + " " + df_teams["NICKNAME"]

# Build lookup dictionary
id_to_name = df_teams.set_index("TEAM_ID")["FULL_NAME"].to_dict()

In [29]:
# Maps the team IDs to team names to improve readability
latest_stats = games.groupby("TEAM_ID").last().reset_index()
latest_stats["TEAM_NAME"] = latest_stats["TEAM_ID"].map(id_to_name)

# Saves the dataframe for later use
latest_stats.to_csv("latest_stats.csv", index=False)